# Извлечение сведений и оценка срочности

|  |  |
|---|---|
| Данные | 40 синтетических обращений на русском языке |
| Сравнение | Диалог с пользователем и оценка по фактам из разметки |
| Шкала | От L0 (самопомощь) до L3 (срочная помощь) |

В рассказе «плохо сплю уже несколько недель» нужно распознать жалобу и понять, как долго она беспокоит человека. Если сервис ошибётся в жалобе или длительности, могут сработать другие правила. Сравним сведения и уровень срочности после диалога с разметкой. Затем передадим размеченные факты напрямую тем же правилам и сверим полученный уровень.

## 1. Разметка обращений

В каждом примере есть рассказ, ответы о длительности, частоте и влиянии на жизнь, а также ответ на вопрос о самоповреждении. Срочность размечена по определениям L0–L3 без экспертной проверки. Таблица показывает, сколько обращений приходится на каждый уровень.

In [ ]:
from collections.abc import Mapping
from pathlib import Path
import sys
from typing import cast

from dishka import AsyncContainer, Provider
from IPython.display import display
import numpy as np
import pandas as pd

from anamnesis.application.dto.views.summary import CaseView, SummaryView
from anamnesis.application.interfaces.knowledge.vocabulary import IOntologyVocabulary
from anamnesis.application.interfaces.llm.language_model import ILanguageModel
from anamnesis.domain.enums.recommendation import UrgencyLevel
from anamnesis.domain.types import TermId
from anamnesis.main.config.config import Config
from anamnesis.main.di.container import create_container


if Path.cwd().name == "benchmarks":
    sys.path.insert(0, str(Path.cwd().parent))

from benchmarks.utils.extraction import (
    Appeal,
    Evaluation,
    ModelCalls,
    count_calls,
    evaluate,
    load_appeals,
)
from benchmarks.utils.support import make_table, use_project_root


use_project_root()
config: Config = Config.load()
cases: list[Appeal] = load_appeals()
pd.Series([case.level for case in cases]).value_counts().sort_index().to_frame("Обращений")

,Обращений
L0,10
L1,12
L2,12
L3,6


## 2. Оценка срочности

Для каждого обращения начинаем отдельный чат и отвечаем на вопросы по разметке. Если ответа нет, пропускаем вопрос. Если сервис обнаружит кризисный сигнал, он может перейти к сводке сразу после рассказа. Во втором прогоне передаём проявления, стрессоры и шкалы из разметки напрямую в онтологию. В обоих случаях правила обрабатывает Pellet.

In [ ]:
monitor: Provider = Provider()
monitor.decorate(count_calls)

In [3]:
app: AsyncContainer
scope: AsyncContainer
case: Appeal
results: list[Evaluation] = []

async with create_container(config, monitor) as app:
    model: ModelCalls = cast(ModelCalls, await app.get(ILanguageModel))
    vocabulary: IOntologyVocabulary = await app.get(IOntologyVocabulary)
    concepts: Mapping[TermId, TermId] = vocabulary.vocabulary().concepts

    for case in cases:
        async with app() as scope:
            results.append(await evaluate(scope, case))

make_table(
    {
        "Модель": config.llm.model,
        "Обращений": len(results),
        "Запросов на извлечение": model.extractions,
        "Ответов вне схемы": model.invalid_extractions,
    }
)

,Модель,Обращений,Запросов на извлечение,Ответов вне схемы
0,qwen3.5:9b,40,38,0


## 3. Оценка извлечения

Для каждого обращения сопоставим извлечённые понятия с обязательными метками. Допустимые дополнительные метки перечислены в поле `optional`. За них балл не меняется. Точность показывает долю верных находок среди извлечённых, полнота — долю найденных среди обязательных. Для F1 суммируем верные находки, лишние понятия и пропуски по всем обращениям.

Длительность сравниваем по границе в две недели, которую используют правила. Для частоты и влияния требуется совпадение значения шкалы. Четырём значениям частоты от редкой до постоянной соответствуют числа от 0 до 3. Если диалог завершится до уточнения, часть шкал может остаться пустой. Доли совпадений считаем по всем 40 обращениям.

In [ ]:
type ScaleComparison = tuple[str, int | None, int]

tp: int = 0
fp: int = 0
fn: int = 0
scales: list[ScaleComparison] = []
rows: list[dict[str, str]] = []
result: Evaluation

for result in results:
    case: Appeal = result.appeal
    summary: SummaryView = result.summary
    facts: CaseView = summary.case

    expected_terms: set[str] = set(case.manifestations + case.stressors)
    found: set[TermId] = {
        concepts.get(evidence.term, evidence.term)
        for evidence in (*facts.manifestations, *facts.stressors)
    }
    missing: set[str] = expected_terms - found
    tp += len(found & expected_terms)
    fp += len(found - expected_terms - set(case.optional))
    fn += len(missing)

    duration: bool | None = None if facts.duration_weeks is None else facts.duration_weeks >= 2
    case_scales: list[ScaleComparison] = [
        ("Длительность ≥ 2 недель", duration, case.duration_weeks >= 2),
        ("Частота", facts.frequency, case.frequency),
        ("Влияние", facts.impact, case.impact),
    ]
    scales.extend(case_scales)

    differences: list[str] = []
    scale_name: str
    scale_value: int | None
    target: int
    for scale_name, scale_value, target in case_scales:
        if scale_value != target:
            value: str | int = "нет" if scale_value is None else int(scale_value)
            differences.append(f"{scale_name}: {int(target)} → {value}")

    crisis_signals: set[str] = {
        signal.category.value for signal in summary.signals if signal.category.is_crisis
    }
    reference_level: UrgencyLevel | None = result.reference_level
    rows.append(
        {
            "Пример": case.id,
            "Эталон": case.level,
            "Диалог": summary.level.name if summary.level is not None else "Нет уровня",
            "По разметке": reference_level.name if reference_level is not None else "Нет уровня",
            "Шкалы (эталон → диалог)": ", ".join(differences),
            "Кризисные сигналы": ", ".join(sorted(crisis_signals)),
            "Пропущенные понятия": ", ".join(sorted(missing)),
        }
    )

display(
    make_table(
        {
            "Верные находки": tp,
            "Лишние": fp,
            "Пропуски": fn,
            "Точность": tp / (tp + fp) if tp + fp else 0,
            "Полнота": tp / (tp + fn),
            "F1": 2 * tp / (2 * tp + fp + fn),
        },
        decimals=3,
    )
)

scale_table: pd.DataFrame = pd.DataFrame(scales, columns=["Сведение", "Получено", "Эталон"])
scale_table["Совпало"] = scale_table["Получено"].eq(scale_table["Эталон"])
scale_table.groupby("Сведение").agg(
    Обращений=("Эталон", "size"),
    Заполнено=("Получено", "count"),
    Совпало=("Совпало", "sum"),
    Доля=("Совпало", "mean"),
).round(3)

,Верные находки,Лишние,Пропуски,Точность,Полнота,F1
0,54,16,11,0.771,0.831,0.8


,Обращений,Заполнено,Совпало,Доля
Сведение,,,,
Влияние,40,35,34,0.850
Длительность ≥ 2 недель,40,35,31,0.775
Частота,40,34,32,0.800


## 4. Расхождения

В матрице строка соответствует эталону, столбец — уровню после диалога. Ошибки слева от диагонали означают, что система занизила срочность. Отдельный столбец «Нет уровня» учитывает случаи, в которых сведений не хватило для оценки. Для обращений с расхождением показываем различия в шкалах и кризисные сигналы. У длительности 1 означает, что пройдена граница в две недели, 0 — что срок меньше.

In [5]:
levels: list[str] = ["L0", "L1", "L2", "L3"]
report: pd.DataFrame = pd.DataFrame(rows).set_index("Пример")
matrix: pd.DataFrame = pd.crosstab(report["Эталон"], report["Диалог"])
display(matrix.reindex(index=levels, columns=[*levels, "Нет уровня"], fill_value=0))

rank: dict[str, int] = {level: i for i, level in enumerate(levels)}
expected: pd.Series = report["Эталон"].map(rank)
comparison: dict[str, dict[str, int | np.int64]] = {}
name: str
for name in ("Диалог", "По разметке"):
    actual: pd.Series = report[name].map(rank)
    comparison[name] = {
        "Совпало": actual.eq(expected).sum(),
        "Занижено": actual.lt(expected).sum(),
        "Завышено": actual.gt(expected).sum(),
        "Нет уровня": actual.isna().sum(),
    }

display(pd.DataFrame(comparison).T)
disagrees: pd.Series = report["Диалог"].ne(report["Эталон"]) | report["По разметке"].ne(
    report["Эталон"]
)
report.loc[disagrees]

Диалог,L0,L1,L2,L3,Нет уровня
Эталон,,,,,
L0,9,0,0,0,1
L1,0,10,2,0,0
L2,0,1,10,1,0
L3,0,0,0,6,0


,Совпало,Занижено,Завышено,Нет уровня
Диалог,35,1,3,1
По разметке,40,0,0,0


,Эталон,Диалог,По разметке,Шкалы (эталон → диалог),Кризисные сигналы,Пропущенные понятия
Пример,,,,,,
a02,L0,Нет уровня,L0,Частота: 1 → нет,,
a14,L1,L2,L1,,,Irritability_
a18,L1,L2,L1,,,
a25,L2,L1,L2,Длительность ≥ 2 недель: 1 → 0,,
a33,L2,L3,L2,"Длительность ≥ 2 недель: 1 → нет, Частота: 1 →...",indirect,"Anxiety_, OngoingDomesticViolence_"


## Выводы

1. Система извлекла 54 из 65 обязательных понятий и добавила 16 лишних (F1 = 0,80). Значение частоты совпало с разметкой в 32 обращениях из 40. В двух случаях сервис записал другое значение, в шести частота осталась неизвестной.
2. После диалога уровень совпал с эталоном в 35 из 40 случаев. Все шесть обращений L3 получили срочную оценку. Единственное занижение пришлось на `a25`. Система нашла обе обязательные жалобы, но отнесла длительность к сроку меньше двух недель. В разметке указаны три недели. В `a02` для оценки L0 не хватило частоты.
3. Когда правила получили факты из разметки, совпали все 40 уровней. Это проверка согласованности правил с определениями L0–L3 на синтетических примерах. В диалоге `a33` детектор нашёл косвенный кризисный сигнал, диалог завершился до уточнения шкал и выдал L3 при эталоне L2.